# Carga de desembolsos

Version corregida del notebook de prueba. Cumple las reglas obligatorias
del Checklist v4 que evalua la capa determinista.

## 1. Cabecera
> **Descripcion:** Informacion general del proceso.

In [ ]:
# ---------------------------------------------------------------------
# PROYECTO       : Operaciones - Desembolsos
# OBJETIVO       : Consolidar los desembolsos del dia
# VERSION        : 1.0.0
# DESARROLLADOR  : Eduardo Fajardo
# FECHA          : 30/08/2026
# TABLA FUENTE   : ope.h_desembolso
# TABLA DESTINO  : operaciones.fct_desembolso
# ---------------------------------------------------------------------

## 2. Importacion de librerias
> **Descripcion:** Primero las librerias estandar y despues las de terceros.

In [ ]:
import logging
import time

from pyspark.sql import functions as F
from pyspark.sql.types import DateType, StringType, StructField, StructType

## 3. Lectura de parametros
> **Descripcion:** Todo parametro llega por widgets, sin valores fijos.

In [ ]:
dbutils.widgets.text("p_catalogo", "")
dbutils.widgets.text("p_fecha_proceso", "")

var_catalogo = dbutils.widgets.get("p_catalogo")
var_fecha_proceso = dbutils.widgets.get("p_fecha_proceso")

logger = logging.getLogger("ETL_DESEMBOLSOS")
logger.setLevel(logging.INFO)

ini_proceso = time.perf_counter()
logger.info("Inicio del proceso ETL_DESEMBOLSOS")
logger.info("Parametros de los widgets: catalogo=%s fecha=%s",
            var_catalogo, var_fecha_proceso)

## 4. Seccion constantes
> **Descripcion:** Rutas y nombres derivados de los parametros recibidos.

In [ ]:
TBL_ORIGEN = f"{var_catalogo}.ope.h_desembolso"
TBL_DESTINO = f"{var_catalogo}.operaciones.fct_desembolso"

ESQUEMA_DESEMBOLSO = StructType([
    StructField("cod_operacion", StringType()),
    StructField("fec_desembolso", DateType()),
])

## 5. Funciones de transformacion
> **Descripcion:** Funciones nativas de PySpark, sin UDF ni SQL en texto.

In [ ]:
def read_desembolso(tabla, fecha):
    """Lee los desembolsos de la fecha indicada.

    Proyecta unicamente las columnas que el proceso necesita.
    """
    return (
        spark.table(tabla)
        .select("cod_operacion", "cod_cliente", "mto_desembolso",
                "fec_desembolso", "tip_moneda")
        .filter(F.col("fec_desembolso") == fecha)
    )


def add_moneda_normalizada(df_origen):
    """Normaliza el codigo de moneda con funciones nativas."""
    return df_origen.withColumn("tip_moneda", F.upper(F.col("tip_moneda")))

## 6. Logica del proceso
> **Descripcion:** Orquestacion de las funciones definidas antes.

In [ ]:
ini_etapa = time.perf_counter()

df_desembolso = read_desembolso(TBL_ORIGEN, var_fecha_proceso)
df_desembolso_final = add_moneda_normalizada(df_desembolso)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 7. Escritura en la tabla final
> **Descripcion:** Persistencia en Delta, sin opciones que alteren la estructura.

In [ ]:
ini_escritura = time.perf_counter()

try:
    (
        df_desembolso_final
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_DESTINO)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_DESTINO, exc)
    raise

logger.info("Tiempo de escritura: %.2f segundos",
            time.perf_counter() - ini_escritura)

## 8. Registro de la ejecucion
> **Descripcion:** Cierre del proceso con la duracion total.

In [ ]:
logger.info("Fin del proceso ETL_DESEMBOLSOS. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)